In [ ]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from lightgbm import LGBMRegressor # Dùng Regressor cho Revenue
from sklift.models import SoloModel

import sys
sys.path.append("..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data...")
train_df = pd.read_csv(r"../dataset/Hillstrom/Men/train_men.csv")
test_df =  pd.read_csv(r"../dataset/Hillstrom/Men/test_men.csv")
val_df = pd.read_csv(r"../dataset/Hillstrom/Men/val_men.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'spend' # BÀI TOÁN REVENUE (Biến liên tục)
treatment_feature = 'treatment'

X_train = train_df[in_features]
y_train = train_df[label_feature]
t_train = train_df[treatment_feature]

X_val = val_df[in_features]
y_val_true = val_df[label_feature].values.flatten()
t_val_true = val_df[treatment_feature].values.flatten()

X_test = test_df[in_features]
y_test_true = test_df[label_feature].values.flatten()
t_test_true = test_df[treatment_feature].values.flatten()

# Danh sách 5 seeds
seeds = [412312, 42, 1874, 902745, 1]

# Cố định seed cho môi trường chung (Dùng seed đầu tiên làm seed chính cho sampler)
seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective (TÍNH TRUNG BÌNH 5 SEEDS)
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'verbose': -1
    }
    
    val_auqc_scores = []
    
    # Duyệt qua 5 seeds BÊN TRONG hàm objective
    for SEED in seeds:
        params['random_state'] = SEED
        
        # SỬ DỤNG LGBMRegressor CHO REVENUE
        base_model = LGBMRegressor(**params)
        s_learner = SoloModel(estimator=base_model)
        
        s_learner.fit(X=X_train, y=y_train, treatment=t_train)
        uplift_pred_val = s_learner.predict(X_val)
        
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    # Trả về AUQC trung bình của 5 seeds
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (Tối ưu điểm trung bình 5 seeds cho REVENUE)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="S_Learner_Robust_Revenue_Tuning", sampler=fixed_sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")
print("Bộ tham số Robust tốt nhất:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# 4. Đánh giá bộ tham số tốt nhất trên tập TEST cho cả 5 seeds
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ BEST PARAMS TRÊN TẬP TEST (REVENUE)")
print("="*50)

test_results = []

for SEED in seeds:
    # Set seed cho best params
    best_params_final = best_params.copy()
    best_params_final['random_state'] = SEED
    best_params_final['verbose'] = -1
    
    final_base_model = LGBMRegressor(**best_params_final)
    final_s_learner = SoloModel(estimator=final_base_model)
    final_s_learner.fit(X=X_train, y=y_train, treatment=t_train)
    
    uplift_pred_test = final_s_learner.predict(X_test)
    
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.3f}, AUQC={auqc_score:.3f}, Lift={lift_score:.3f}, KRCC={krcc_score:.3f}")

# Tính toán điểm số trung bình trên tập Test
results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()

print("\n" + "*"*50)
print("🏆 KẾT QUẢ TRUNG BÌNH TRÊN TẬP TEST (5 SEEDS - REVENUE) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f}")

# Lưu kết quả chi tiết
csv_filename = "s_learner_revenue_robust_test_results.csv"
results_df.to_csv(csv_filename, index=False)

Loading data...
Locked random seed: 412312

🔃 Đang chạy Optuna Tuning (Tối ưu điểm trung bình 5 seeds cho REVENUE)...


  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 35. Best value: 0.556005: 100%|██████████| 50/50 [02:07<00:00,  2.54s/it]


✅ Tuning hoàn tất! Best Mean Val AUQC: 0.5560
Bộ tham số Robust tốt nhất:
   n_estimators: 51
   learning_rate: 0.026518989569493133
   max_depth: 9
   num_leaves: 127
   min_child_samples: 46
   subsample: 0.9656012518610151
   colsample_bytree: 0.9626626186823356

🚀 ĐÁNH GIÁ BEST PARAMS TRÊN TẬP TEST (REVENUE)
Seed 412312: AUUC=0.622, AUQC=0.620, Lift=0.645, KRCC=0.049
Seed 42: AUUC=0.622, AUQC=0.620, Lift=0.645, KRCC=0.049
Seed 1874: AUUC=0.622, AUQC=0.620, Lift=0.645, KRCC=0.049
Seed 902745: AUUC=0.622, AUQC=0.620, Lift=0.645, KRCC=0.049
Seed 1: AUUC=0.622, AUQC=0.620, Lift=0.645, KRCC=0.049

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH TRÊN TẬP TEST (5 SEEDS - REVENUE) 🏆
**************************************************
Mean AUUC: 0.622
Mean AUQC: 0.620
Mean Lift: 0.645
Mean KRCC: 0.049
